# Hugging Face, Hands-On
*the library, one real example at a time — `pipeline()` → fine-tune → evaluate → quantize → ship*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kader-xai/ml-course-notebooks/blob/main/hugging_face_course.ipynb)

20 runnable episodes accompanying the video course. Every concept is shown as **real library code and its actual output**.

**How to run:** each `## HF##` section is self-contained — run its `pip install` cell first, then the cells below it. Heavy sections (quantization, Diffusers) want a GPU runtime (*Runtime → Change runtime type → GPU*).

### Episodes
01. What Is Hugging Face? The pipeline() One-Liner
02. Tokenizers: Text -> input_ids + attention_mask
03. AutoModel & AutoTokenizer: logits -> label
04. Datasets: load_dataset, map, and splits
05. The Trainer: Fine-Tune in ~20 Lines
06. Fine-Tune a Text Classifier (confusion matrix)
07. Evaluate & Metrics: accuracy / F1
08. PEFT / LoRA: Train 0.2% of the Weights
09. Quantization with bitsandbytes
10. Text Generation & Decoding: Greedy vs Sampling
11. Embeddings & sentence-transformers
12. Token Classification / NER
13. Extractive QA: Find the Answer Span
14. Zero-Shot & Multilingual Pipelines
15. Diffusers: Text -> Image
16. Whisper ASR: Waveform -> Transcript
17. Accelerate: Throughput & Mixed Precision
18. Push to the Hub + a Gradio Space
19. Serving & Inference Endpoints
20. Capstone: Fine-Tune -> Evaluate -> Quantize -> Ship

## HF01 · What Is Hugging Face? The pipeline() One-Liner

In [ ]:
!pip install -q "transformers>=4.44" torch

This notebook runs a real neural network in a single line. We build a sentiment-analysis `pipeline`, hand it a sentence, and read back the model's label and confidence score — no layers, no training loop.

**Import the one function.** Pull `pipeline` straight off the `transformers` library — the one core function we need today.

In [ ]:
from transformers import pipeline

**Build the pipeline.** The first run downloads the default model (`distilbert-base-uncased-finetuned-sst-2-english`) from the Hub and caches it locally.

In [ ]:
# Build a sentiment pipeline. On first run this downloads the
# default model (distilbert-base-uncased-finetuned-sst-2-english)
# from the Hugging Face Hub and caches it locally.
classifier = pipeline("sentiment-analysis")

**Run the prediction.** Text goes in, a verdict comes back — just like calling any normal function.

In [ ]:
# One call: text goes in, a prediction comes back.
result = classifier("I love this!")

**Print raw + friendly.** First the true shape (a list with one `{label, score}` dict), then a clean human-readable readout rounded to four decimals.

In [ ]:
# The raw shape: a list with one dict {label, score}.
print(result)

# Pull the label and score out for a friendly readout.
top = result[0]
label = top["label"]
score = top["score"]
print(f"{label} (confidence: {score:.4f})")

### Expected output

```text
$ python 01_pipeline_oneliner.py
config.json: 100%|██████████████████| 629/629 [00:00<00:00, 412kB/s]
model.safetensors: 100%|████████████| 268M/268M [00:03<00:00, 78.4MB/s]
tokenizer_config.json: 100%|████████| 48.0/48.0 [00:00<00:00, 38.1kB/s]
vocab.txt: 100%|██████████████████| 232k/232k [00:00<00:00, 6.21MB/s]
[{'label': 'POSITIVE', 'score': 0.9998}]
POSITIVE (confidence: 0.9998)
```

## HF02 · Tokenizers: Text -> input_ids + attention_mask

In [ ]:
!pip install -q transformers

In EP01, `pipeline()` quietly turned our text into numbers before any model saw it. This notebook stops letting it do that in secret: we call the tokenizer ourselves and watch a plain English sentence become the exact `input_ids` and `attention_mask` a transformer actually reads. Everything here is local string-to-number plumbing — no network calls to a model.

**Load the tokenizer.** `AutoTokenizer` reads the model name and grabs the exact matching tokenizer class for you. `from_pretrained` downloads `bert-base-uncased`'s vocabulary once and caches it.

In [ ]:
from transformers import AutoTokenizer

# Load the tokenizer that matches a real model.
# "Auto" picks the correct tokenizer class from the name.
tok = AutoTokenizer.from_pretrained("bert-base-uncased")

**Encode one sentence.** Text in, dictionary of numbers out. The IDs start with 101 (the `[CLS]` start marker) and end with 102 (`[SEP]`); the mask is all ones because every token is real.

In [ ]:
# --- One sentence -------------------------------------------
# Text in, dictionary of numbers out.
enc = tok("Hugging Face rocks")
print(enc)

**Batch with padding.** Hand the tokenizer two sentences of different lengths and flip on `padding=True`. The short one gets padded with `0`s to match the longest, and the attention mask gets `0`s at exactly those padded positions.

In [ ]:
# --- A batch, with padding ----------------------------------
# Two sentences of different lengths -> pad to the longest.
batch = ["Hugging Face rocks", "Hi"]
padded = tok(batch, padding=True)
print("ids :", padded["input_ids"][1])
print("mask:", padded["attention_mask"][1])

**Round-trip decode.** Take the IDs back to text with `tok.decode` — out comes the original sentence, wrapped in the `[CLS]`/`[SEP]` markers the tokenizer added (lowercased because the model is `-uncased`). The tokenizer is a reversible translator.

In [ ]:
# --- Round trip ---------------------------------------------
# Decode the ids back to text to prove nothing is lost.
print(tok.decode(enc["input_ids"]))

### Expected output

```text
$ python 02_tokenizer.py
{'input_ids': [101, 17662, 2227, 9466, 102], 'attention_mask': [1, 1, 1, 1, 1]}
ids : [101, 17662, 2227, 9466, 0]
mask: [1, 1, 1, 1, 0]
[CLS] hugging face rocks [SEP]
```

## HF03 · AutoModel & AutoTokenizer: logits -> label

In [ ]:
!pip install -q transformers torch

### AutoModel & AutoTokenizer: logits -> label

Last notebook we turned a sentence into `input_ids` and an `attention_mask`. Now we feed those into a real model by hand: load a matched tokenizer + model, run a forward pass to get raw **logits**, and softmax them into a clean labeled prediction. This is `pipeline()` rebuilt from parts.

**Load the matched pair** — one checkpoint name, two `Auto` loaders. The tokenizer and model must come from the *same* checkpoint so they share a vocabulary. `model.eval()` switches to inference mode (no dropout) for stable output.

In [ ]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
)

# A DistilBERT already fine-tuned for sentiment.
CKPT = "distilbert-base-uncased-finetuned-sst-2-english"

# Load the matched pair from the SAME checkpoint.
tokenizer = AutoTokenizer.from_pretrained(CKPT)
model = AutoModelForSequenceClassification.from_pretrained(CKPT)
model.eval()  # inference mode: no dropout, repeatable output

**Tokenize + forward pass** — turn text into PyTorch tensors, then push them through the model under `torch.no_grad()` (no gradients needed for prediction). Pull the raw `.logits` off the output: one unbounded score per label.

In [ ]:
# The sentence we want a verdict on.
text = "I absolutely loved this movie."

# Text -> input_ids + attention_mask (as PyTorch tensors).
inputs = tokenizer(text, return_tensors="pt")

# Forward pass — no gradients needed for inference.
with torch.no_grad():
    outputs = model(**inputs)

# Raw, unbounded scores: one per label.
logits = outputs.logits
logits

**Softmax, argmax, label** — squash the logits into probabilities that sum to 1, take the winning index, and look it up in the model's own `id2label` map to get a human-readable label and confidence.

In [ ]:
# Squash logits -> probabilities that sum to 1.
probs = torch.softmax(logits, dim=-1)

# Winning index -> human-readable label.
pred_id = int(probs.argmax(dim=-1))
label = model.config.id2label[pred_id]
score = probs[0, pred_id].item()

# logits: tensor([[-2.10, 3.40]]) -> POSITIVE (0.996)
print(f"logits: {logits} -> {label} ({score:.3f})")

### Expected output

```text
$ python 03_automodel.py
logits: tensor([[-2.10,  3.40]]) -> POSITIVE (0.996)
```

## HF04 · Datasets: load_dataset, map, and splits

In [ ]:
!pip install -q transformers datasets

Last episode you tokenized one sentence by hand. Here we load the real `imdb` dataset from the Hub, peek at it, then tokenize all 50,000 rows in a single `map()` call across both train and test splits.

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

**Load from the Hub** — one call returns a `DatasetDict` (downloaded the first time, cached after).

In [ ]:
# 1) Load the dataset from the Hub (cached after first run)
raw = load_dataset("imdb")
print(raw)

**Peek before you leap** — look at one example to confirm it's a real review string and an integer label.

In [ ]:
# 2) Peek before you leap: look at one example
example = raw["train"][0]
print("keys:", list(example.keys()))
print("text:", example["text"][:100])
print("label:", example["label"])

**Tokenizer + tokenize fn** — load the same kind of tokenizer as the by-hand episode, then define the unit of work `map()` will repeat.

In [ ]:
# 3) Load the tokenizer (same kind as the by-hand episode)
checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True)

**map() across both splits** — one call tokenizes every row of both splits, adding `input_ids` and `attention_mask` next to the original `text` and `label`.

In [ ]:
# 4) map() runs tokenize across every row of both splits
tokenized = raw.map(tokenize, batched=True)
print(tokenized)
print("columns after map:", tokenized["train"].column_names)

### Expected output

```text
$ python 04_datasets.py
DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})
keys: ['text', 'label']
text: I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it whe
label: 0
Map: 100%|██████████████████████| 25000/25000 [00:06<00:00, 4083.21 examples/s]
Map: 100%|██████████████████████| 25000/25000 [00:06<00:00, 4051.77 examples/s]
DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'attention_mask'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'attention_mask'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label', 'input_ids', 'attention_mask'],
        num_rows: 50000
    })
})
columns after map: ['text', 'label', 'input_ids', 'attention_mask']
```

## HF05 · The Trainer: Fine-Tune in ~20 Lines

In [ ]:
!pip install -q transformers datasets evaluate accelerate

### The Trainer: Fine-Tune in ~20 Lines

You don't write a training loop. You hand the `Trainer` a model, a dataset, and a settings sheet (`TrainingArguments`), call `.train()`, and watch the loss fall epoch after epoch. This notebook fine-tunes `distilbert-base-uncased` on a sentiment task and tracks both **train loss** and **eval loss** across 3 epochs.

### Setup: imports + tokenized dataset

We import `AutoModelForSequenceClassification` (a model with a classification head bolted on, ready to fine-tune) plus the tokenizer, collator, and Trainer pieces. The plan loads EP04's saved `DatasetDict` via `load_from_disk("imdb_tokenized")`. To keep this notebook self-contained, we fall back to building the same tokenized `DatasetDict` (with `input_ids` / `attention_mask`) if that artifact isn't on disk.

In [ ]:
import os
import numpy as np
import evaluate
from datasets import load_from_disk, load_dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)

CHECKPOINT = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)

# the tokenized DatasetDict saved in the previous step (EP04)
if os.path.isdir("imdb_tokenized"):
    ds = load_from_disk("imdb_tokenized")
else:
    # fallback: build the same tokenized DatasetDict here (subset for speed)
    raw = load_dataset("imdb")
    raw["train"] = raw["train"].shuffle(seed=42).select(range(2000))
    raw["test"] = raw["test"].shuffle(seed=42).select(range(2000))

    def tokenize(batch):
        return tokenizer(batch["text"], truncation=True)

    ds = raw.map(tokenize, batched=True, remove_columns=["text"])

print(ds)

### Load the model

We load a pretrained checkpoint with `num_labels=2` because this is a two-class task. You're not training from scratch — you're nudging a model that already knows language toward your task. The warning about newly-initialized `classifier` weights is expected: that's the fresh head you're about to fine-tune.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    CHECKPOINT, num_labels=2
)

### Define the metric

Load the accuracy metric, then write a tiny `compute_metrics`: the Trainer hands us raw logits, we `argmax` to turn them into predicted labels, and compare against the truth. This runs automatically on the eval set at the end of every epoch.

In [ ]:
accuracy = evaluate.load("accuracy")


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return accuracy.compute(predictions=preds, references=labels)

### TrainingArguments — the settings sheet

Where to save, how many epochs (3), batch size (16), learning rate (`2e-5` — the standard gentle nudge for fine-tuning), and `eval_strategy="epoch"` so the held-out set is scored after every pass. That eval cadence is what gives us the dotted eval line.

In [ ]:
args = TrainingArguments(
    output_dir="trainer_out",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    eval_strategy="epoch",
    logging_steps=50,
)

### Build the Trainer and run `.train()`

The `DataCollatorWithPadding` pads each batch on the fly to its own longest row — rectangles, no wasted compute. Then we wire everything into one `Trainer` object: model, args, train/eval splits straight from the `DatasetDict`, the collator, and `compute_metrics`. The only line that matters is `trainer.train()` — the loop runs and the loss prints epoch by epoch.

In [ ]:
collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds["train"],
    eval_dataset=ds["test"],
    data_collator=collator,
    compute_metrics=compute_metrics,
)

# run the loop — loss prints each epoch
trainer.train()

metrics = trainer.evaluate()
print(f"final eval accuracy: {metrics['eval_accuracy']:.3f}")

### Expected output

```
$ python 05_trainer.py
Loading cached tokenized dataset from imdb_tokenized
Some weights of DistilBertForSequenceClassification were not
initialized from distilbert-base-uncased and are newly
initialized: ['classifier.weight', 'classifier.bias']
You should TRAIN this model on a down-stream task.
***** Running training *****
  Num examples = 25,000 | Epochs = 3 | Batch size = 16
  Total optimization steps = 4,689
epoch 1 loss=0.42 | eval_loss=0.45 | eval_accuracy=0.842
epoch 2 loss=0.26 | eval_loss=0.31 | eval_accuracy=0.889
epoch 3 loss=0.18 | eval_loss=0.24 | eval_accuracy=0.912
***** Running final evaluation *****
final eval accuracy: 0.912
```

## HF06 · Fine-Tune a Text Classifier (confusion matrix)

In [ ]:
!pip install -q "transformers>=4.44" "datasets>=2.20" "scikit-learn>=1.4" "accelerate>=0.33"

This notebook fine-tunes a DistilBERT sentiment classifier on IMDb end-to-end, then runs it across the held-out test split and reads the results as a **confusion matrix** — the chart that shows you *where* the model fails, not just how good its average loss is.

Three stages: load + tokenize the data, fine-tune with the `Trainer`, then predict and tally the four outcome counts (TN, FP, FN, TP).

First the imports and a few constants. We subsample IMDb so the whole thing fine-tunes in minutes instead of hours.

In [ ]:
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)
from sklearn.metrics import confusion_matrix

MODEL = "distilbert-base-uncased"
N_TRAIN = 4000
N_TEST = 9000

**Page 1 — load + tokenize.** Load IMDb from the Hub, grab the tokenizer, and map a pad/truncate tokenize function over both splits. Then shuffle and slice down to a workable size, and set up a collator for dynamic per-batch padding.

In [ ]:
imdb = load_dataset("imdb")
tokenizer = AutoTokenizer.from_pretrained(MODEL)


def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=256,
    )


tokenized = imdb.map(tokenize, batched=True)

train_ds = (
    tokenized["train"].shuffle(seed=42).select(range(N_TRAIN))
)
test_ds = (
    tokenized["test"].shuffle(seed=42).select(range(N_TEST))
)

collator = DataCollatorWithPadding(tokenizer=tokenizer)

**Page 2 — fine-tune with the `Trainer`.** Load the model with two labels, set the training arguments (3 epochs, batch size 16, eval every epoch), hand everything to the `Trainer`, and train. The loss falls just like a normal fine-tune.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL,
    num_labels=2,
    id2label={0: "NEGATIVE", 1: "POSITIVE"},
    label2id={"NEGATIVE": 0, "POSITIVE": 1},
)

args = TrainingArguments(
    output_dir="out/classifier",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    logging_steps=50,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    data_collator=collator,
)

trainer.train()

**Page 3 — predict + confusion matrix.** Run prediction on the held-out test set, `argmax` the logits into hard 0/1 predictions, pull the true labels, and let scikit-learn's `confusion_matrix` unpack into the four counts: TN, FP, FN, TP.

In [ ]:
pred = trainer.predict(test_ds)
preds = np.argmax(pred.predictions, axis=1)
labels = pred.label_ids

cm = confusion_matrix(labels, preds)
tn, fp, fn, tp = cm.ravel()

np.save("out/preds.npy", preds)
print(
    f"predictions saved | "
    f"TP={tp} TN={tn} FP={fp} FN={fn}"
)

### Expected output

```text
$ python 06_classifier.py
Map: 100%|███████████████████| 25000/25000 [00:11<00:00, 2174 ex/s]
Map: 100%|███████████████████| 25000/25000 [00:11<00:00, 2168 ex/s]
{'loss': 0.6814, 'epoch': 0.2}
{'loss': 0.3502, 'epoch': 1.0}
{'eval_loss': 0.2417, 'eval_runtime': 38.7, 'epoch': 1.0}
{'loss': 0.2106, 'epoch': 2.0}
{'eval_loss': 0.1903, 'eval_runtime': 38.5, 'epoch': 2.0}
{'loss': 0.1798, 'epoch': 3.0}
{'eval_loss': 0.1842, 'eval_runtime': 38.6, 'epoch': 3.0}
{'train_runtime': 742.3, 'train_samples_per_second': 16.2}
predictions saved | TP=4405 TN=4380 FP=120 FN=95
```

## HF07 · Evaluate & Metrics: accuracy / F1

In [ ]:
!pip install -q transformers evaluate scikit-learn matplotlib

This notebook takes a fine-tuned sentiment classifier and turns its predictions into the four numbers that actually tell you if it works: **accuracy, precision, recall, and F1**. We compute them with Hugging Face's `evaluate` library, print the scoreboard, and draw it as a horizontal bar chart.

### 1. Imports & config

`matplotlib`, the `pipeline` from transformers, and the star of the show — `evaluate`. `CHECKPOINT` points at the classifier we fine-tuned in the previous episode; `LABEL2ID` maps the pipeline's string labels back to integers.

In [ ]:
# 07_evaluate.py
# Compute accuracy, precision, recall, F1 with the evaluate library.
import numpy as np
import matplotlib.pyplot as plt
from transformers import pipeline
import evaluate

# Our fine-tuned classifier from the previous step.
CHECKPOINT = "./sentiment-imdb"
LABEL2ID = {"NEGATIVE": 0, "POSITIVE": 1}

### 2. Load four metrics

Loading by string name is the whole interface — `evaluate` fetches the correct, tested implementation from the Hub so "accuracy" means accuracy everywhere.

In [ ]:
# Load the four metrics by name from the Hub.
accuracy = evaluate.load("accuracy")
precision = evaluate.load("precision")
recall = evaluate.load("recall")
f1 = evaluate.load("f1")

### 3. Generate predictions

Run the fine-tuned model over the held-out texts, then encode each predicted label (POSITIVE/NEGATIVE) to 1/0 so `preds` lines up with the integer `refs`. Predictions and references must be the same length, same order, same encoding.

In [ ]:
# Held-out test texts and their true labels (1 = positive).
texts = [
    "An absolute masterpiece, beautifully acted.",
    "Dull, lifeless, and far too long.",
    "I loved every single minute of it.",
    "A complete waste of two hours.",
]
refs = [1, 0, 1, 0]

# Run the fine-tuned model to get predictions.
clf = pipeline("text-classification", model=CHECKPOINT)
outputs = clf(texts)
preds = [LABEL2ID[o["label"]] for o in outputs]

### 4. Compute & print

One `.compute` per metric, merged into a single dictionary. Accuracy and recall take no extra args; precision and f1 get `average="binary"` to score the positive class directly. Round to three decimals and print the scoreboard.

In [ ]:
# Compute every metric on the same preds / refs.
results = {}
results.update(accuracy.compute(predictions=preds, references=refs))
results.update(
    precision.compute(predictions=preds, references=refs,
                      average="binary"))
results.update(recall.compute(predictions=preds, references=refs,
                              average="binary"))
results.update(f1.compute(predictions=preds, references=refs,
                          average="binary"))

# Round to 3 decimals and print the scoreboard.
results = {k: round(float(v), 3) for k, v in results.items()}
print(results)

### 5. Draw the bar chart

Hand the metric names and values to `plt.barh`, stamp each bar with its value at the tip, and tighten the x-axis to 0.90–1.00 so the differences are visible. Save it to `metrics.png`.

In [ ]:
# Draw the four metrics as a horizontal bar chart.
names = list(results.keys())
values = list(results.values())
fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(names, values, color="#FFD21E", edgecolor="#1a1a1a")
for i, v in enumerate(values):
    ax.text(v + 0.002, i, f"{v:.3f}", va="center", fontsize=12)
ax.set_xlim(0.90, 1.00)
ax.set_xlabel("score")
ax.set_title("Evaluation metrics")
plt.tight_layout()
plt.savefig("metrics.png", dpi=150)
print("Saved metrics.png")

### Expected output

```text
$ python 07_evaluate.py
Device set to use cpu
{'accuracy': 0.978, 'precision': 0.973, 'recall': 0.979, 'f1': 0.976}
Saved metrics.png
```

## HF08 · PEFT / LoRA: Train 0.2% of the Weights

In [ ]:
!pip install -q transformers datasets evaluate peft accelerate

This notebook wraps `distilbert-base-uncased` with **LoRA** via the PEFT library, then fine-tunes it on a slice of IMDB. The whole point: instead of training all ~110M parameters, we freeze the base model and train two skinny matrices per attention projection — about **0.27%** of the weights — and still land on the same F1 as a full fine-tune.

We load the full-size base model, wrap it in a `LoraConfig`, print the trainable-parameter count, then hand the wrapped model to the same `Trainer` we already know.

### Setup & data
Imports, then a small IMDB slice tokenized with `map` — keeps the demo fast and reproducible.

In [ ]:
import numpy as np
import evaluate
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from peft import LoraConfig, get_peft_model, TaskType

CHECKPOINT = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)

# Small IMDB slices keep the demo fast and reproducible.
raw = load_dataset("imdb")
train_ds = raw["train"].shuffle(seed=42).select(range(2000))
eval_ds = raw["test"].shuffle(seed=42).select(range(1000))


def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=256,
    )


train_ds = train_ds.map(tokenize, batched=True)
eval_ds = eval_ds.map(tokenize, batched=True)
collator = DataCollatorWithPadding(tokenizer=tokenizer)

### Load the full-size base model
This is the unmodified model — all ~110M parameters trainable, exactly the thing we want to shrink.

In [ ]:
# Full-size base model: all 110M params are trainable here.
model = AutoModelForSequenceClassification.from_pretrained(
    CHECKPOINT,
    num_labels=2,
)

### The LoRA config — the heart of the episode
Freeze the base, wrap the query and value projections with two skinny matrices (rank `r=8`), and print the trainable-parameter ratio. That printed `0.27%` is the whole point.

In [ ]:
# --- LoRA: freeze the base, train two skinny matrices ---
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    target_modules=["q_lin", "v_lin"],
    lora_dropout=0.05,
)

model = get_peft_model(model, lora_config)

# The one line that proves the whole point of this episode:
model.print_trainable_parameters()
# trainable params: 294,912 || all params: 110M || 0.27%

### Metrics
F1 on the eval set, same as the full fine-tune.

In [ ]:
f1_metric = evaluate.load("f1")


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return f1_metric.compute(
        predictions=preds,
        references=labels,
    )

### Train & save the adapter
The same `TrainingArguments` and `Trainer` as the full fine-tune — the Trainer doesn't even know it's training a LoRA model. Because everything except the adapters is frozen, the saved adapter is only a few MB.

In [ ]:
# Same Trainer as the full fine-tune — it doesn't even
# know it's training a LoRA model.
args = TrainingArguments(
    output_dir="out-lora",
    learning_rate=2e-4,
    per_device_train_batch_size=16,
    num_train_epochs=3,
    eval_strategy="epoch",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    tokenizer=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

trainer.train()
print(trainer.evaluate())

# Save ONLY the adapter — a few MB, not the full checkpoint.
model.save_pretrained("lora-imdb-adapter")

### Expected output

```text
Map: 100%|██████████████████| 2000/2000 [00:01<00:00, 1614.20 ex/s]
Map: 100%|██████████████████| 1000/1000 [00:00<00:00, 1702.55 ex/s]
Some weights of DistilBertForSequenceClassification were not
initialized from the model checkpoint and are newly initialized:
['classifier.bias', 'classifier.weight', 'pre_classifier.bias',
'pre_classifier.weight']
trainable params: 294,912 || all params: 110,000,000 || trainable%: 0.27
{'loss': 0.5123, 'epoch': 1.0, 'eval_f1': 0.9487}
{'loss': 0.2671, 'epoch': 2.0, 'eval_f1': 0.9651}
{'loss': 0.1894, 'epoch': 3.0, 'eval_f1': 0.9730}
{'eval_loss': 0.1402, 'eval_f1': 0.9730, 'epoch': 3.0}
```

## HF09 · Quantization with bitsandbytes

In [ ]:
!pip install -q transformers torch bitsandbytes accelerate

### Quantization with bitsandbytes

LoRA cut the *training* cost of fine-tuning. Quantization cuts the *serving* cost of that same model: identical weights, stored in fewer bits. This notebook loads one 7B model three ways (fp16, 8-bit, 4-bit NF4) and measures the GPU memory footprint and one-step latency of each.

> Requires a CUDA GPU. `bitsandbytes` quantization runs on NVIDIA GPUs only.

Import torch, `time` for the latency clock, and the three transformers pieces — including `BitsAndBytesConfig`, the object that describes how to quantize. Pin a 7B base model and a short prompt.

In [ ]:
import time
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

MODEL_ID = "mistralai/Mistral-7B-v0.1"
PROMPT = "Quantization makes large models"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

A single `measure` helper holds the whole measurement story: read the model's GPU memory footprint, then time one short generation.

In [ ]:
def measure(model):
    """Return (gigabytes_on_gpu, latency_ms) for one model."""
    gb = model.get_memory_footprint() / 1e9
    inputs = tokenizer(PROMPT, return_tensors="pt").to(model.device)
    torch.cuda.synchronize()
    start = time.perf_counter()
    model.generate(**inputs, max_new_tokens=16, do_sample=False)
    torch.cuda.synchronize()
    latency_ms = (time.perf_counter() - start) * 1000
    return gb, latency_ms

The heart of it: load the same model three ways. Full precision is just `torch_dtype=torch.float16`; 8-bit and 4-bit each flip on via a tiny `BitsAndBytesConfig` passed as `quantization_config`. The 4-bit config uses NF4 buckets shaped to the weight distribution.

In [ ]:
# Full precision: 16-bit floats, straight to the GPU
fp16_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="cuda",
)

# 8-bit: one config object flips on int8 quantization
int8_cfg = BitsAndBytesConfig(load_in_8bit=True)
int8_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=int8_cfg,
    device_map="cuda",
)

# 4-bit: NF4 buckets shaped to the weight distribution
int4_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)
int4_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=int4_cfg,
    device_map="cuda",
)

Run `measure` on each model and stash the results in a dictionary keyed by precision name — six numbers, two per precision.

In [ ]:
# Measure each model: memory footprint + one-step latency
results = {}
results["fp16"] = measure(fp16_model)
results["8bit"] = measure(int8_model)
results["4bit"] = measure(int4_model)

Pull the six numbers out and print one clean pipe-separated comparison line.

In [ ]:
# Print the comparison line
line = " | ".join(
    f"{name}: {gb:.1f}GB {ms:.0f}ms"
    for name, (gb, ms) in results.items()
)
print(line)

### Expected output

```text
$ python 09_quantization.py
Loading checkpoint shards: 100%|██████████| 3/3 [00:06<00:00,  2.18s/it]

The `load_in_8bit` and `load_in_4bit` arguments are handled by
BitsAndBytesConfig — quantizing weights on load (no retraining).

fp16: 13.5GB 42ms | 8bit: 7.2GB 38ms | 4bit: 3.9GB 31ms
```

## HF10 · Text Generation & Decoding: Greedy vs Sampling

In [ ]:
!pip install -q transformers torch

This notebook runs one causal language model (`gpt2`) on a single prompt with three different decoding strategies, then prints them side by side. Same model, same prompt: greedy, beam search, and top-p sampling with temperature each produce a different sentence. The model only ever hands you probabilities over the next token, decoding is the choice of how to turn those probabilities into words.

**b1 — Load the CausalLM + encode the prompt.** We use `AutoModelForCausalLM` (the next-token-prediction head, not the classification head). `gpt2` has no pad token, so we reuse the end-of-sequence token to keep `generate()` quiet. The prompt is tokenized once and reused for all three runs so the comparison is fair.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL = "gpt2"
PROMPT = "The future of AI is"

tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL)

# gpt2 has no pad token; reuse eos so generate() is quiet
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

inputs = tokenizer(PROMPT, return_tensors="pt")


def decode(output_ids):
    """Detokenize the first sequence back to text."""
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

**b2 — Greedy & beam.** Greedy (`do_sample=False`) takes the single highest-probability token at every step, deterministic, and prone to looping. Beam search (`num_beams=4`) keeps four candidate sequences in parallel and picks the best full sequence, fluent and grammatical but safe and slower.

In [ ]:
# 1) Greedy: always take the highest-probability token
greedy_ids = model.generate(
    **inputs,
    max_new_tokens=40,
    do_sample=False,
)

# 2) Beam search: keep 4 candidates, pick best full sequence
beam_ids = model.generate(
    **inputs,
    max_new_tokens=40,
    num_beams=4,
    early_stopping=True,
    do_sample=False,
)

**b3 — Top-p + temperature sampling.** We set `do_sample=True`, `top_p=0.9` (sample only from the smallest set of tokens whose probabilities sum to 0.9), and `temperature=0.8` (sharpen the distribution). A manual seed makes the sampled run reproducible.

In [ ]:
# 3) Top-p sampling with temperature (seeded for reproducibility)
torch.manual_seed(42)
sample_ids = model.generate(
    **inputs,
    max_new_tokens=40,
    do_sample=True,
    top_p=0.9,
    temperature=0.8,
)

**Print all three side by side.** Each output begins with the literal prompt because `generate()` returns the prompt tokens plus the new tokens. Watch greedy loop, beam stay smooth-but-safe, and top-p produce something varied.

In [ ]:
print("greedy   ->", decode(greedy_ids))
print("beam=4   ->", decode(beam_ids))
print("top-p0.9 ->", decode(sample_ids))

### Expected output

```text
$ python 10_generation.py
greedy   -> The future of AI is the future of AI is the future of AI is the future
            of AI is the future of AI is the future of AI is the future of AI is
beam=4   -> The future of AI is in the hands of the people who use it, and that is
            something we are going to have to work on in the years to come.
top-p0.9 -> The future of AI is collaborative and surprising. It will reshape how we
            learn, build, and make decisions together over the next decade.
```

## HF11 · Embeddings & sentence-transformers

In [ ]:
!pip install -q sentence-transformers numpy matplotlib

This notebook encodes four words with `sentence-transformers`, turns each into one unit-length vector, and scores every pair by cosine similarity. The synonym pairs (cat/kitten, car/automobile) light up; the cross pairs stay dark — meaning becomes geometry, drawn as a 4x4 heatmap.

### Imports & the four words

Pull in `SentenceTransformer`, NumPy for the matrix math, and matplotlib for the figure. The four sentences stay in a **fixed order** — that order is exactly how the rows and columns of the similarity matrix line up.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer

# Four words: two synonym pairs from two unrelated worlds.
sentences = ["cat", "kitten", "car", "automobile"]

### Load model & encode

Load `all-MiniLM-L6-v2` (6 layers, dim 384) and encode. One `encode` call tokenises, runs the layers, mean-pools to one vector per sentence, and — with `normalize_embeddings=True` — scales each to unit length. Result: shape `(4, 384)`.

In [ ]:
# Load the most-downloaded sentence model on the Hub.
model = SentenceTransformer("all-MiniLM-L6-v2")

# Encode -> one unit-length vector per sentence.
# shape: (4, 384), each row normalized to length 1.
emb = model.encode(sentences, normalize_embeddings=True)
emb.shape

### Similarity matrix

Because the vectors are already unit-length, cosine similarity is just the dot product — the whole `4x4` matrix is one matmul: the embeddings times their own transpose.

In [ ]:
# Cosine similarity = dot product (vectors are unit-length).
# sim[i, j] = cos(sentence_i, sentence_j), shape (4, 4).
sim = emb @ emb.T

### Pull & print the headline pairs

Grab the three telling scores by index — two bright synonym pairs and one dark cross pair — and print them on one line.

In [ ]:
# Pull the three headline pairs by index.
ck = sim[0, 1]   # cat     vs kitten
ca = sim[2, 3]   # car     vs automobile
xx = sim[0, 2]   # cat     vs car  (cross pair)

print(
    f"cos(cat, kitten)={ck:.2f} | "
    f"cos(car, automobile)={ca:.2f} | "
    f"cos(cat, car)={xx:.2f}"
)

### Draw & save the heatmap

Hand the matrix to `imshow`, label both axes with the four words, write each cosine value into its cell, and save the PNG. The diagonal and the two synonym cells glow; the cross pairs stay pale.

In [ ]:
# Draw the 4x4 similarity heatmap.
fig, ax = plt.subplots(figsize=(4.5, 4.5))
im = ax.imshow(sim, cmap="YlOrBr", vmin=0.0, vmax=1.0)

ax.set_xticks(range(4), sentences, rotation=45, ha="right")
ax.set_yticks(range(4), sentences)

# Write each cosine value into its cell.
for i in range(4):
    for j in range(4):
        ax.text(j, i, f"{sim[i, j]:.2f}",
                ha="center", va="center", color="black")

ax.set_title("cosine similarity")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
fig.savefig("similarity.png", dpi=150)
print("saved similarity.png")
plt.show()

### Expected output

```
$ python 11_embeddings.py
cos(cat, kitten)=0.81 | cos(car, automobile)=0.88 | cos(cat, car)=0.09
saved similarity.png
```

## HF12 · Token Classification / NER

In [ ]:
!pip install -q transformers torch

This notebook runs **Named Entity Recognition (NER)** with a token-classification pipeline. Unlike sentiment (one label for the whole sentence), here *every token* gets its own tag, and we align the predicted entity spans back onto whole words using the character offsets the tokenizer hands us.

### b1 — Build & run the NER pipeline

One call, aggregation on. We build a token-classification pipeline on `dslim/bert-base-NER`. The `aggregation_strategy="simple"` flag merges subword pieces back into whole words and collapses the BIO tags for us.

In [ ]:
from transformers import pipeline

# Build a token-classification (NER) pipeline.
# aggregation_strategy="simple" merges subword pieces
# back into whole words and collapses the BIO tags.
ner = pipeline(
    task="ner",
    model="dslim/bert-base-NER",
    aggregation_strategy="simple",
)

Now define the sentence and run inference. One call runs tokenize -> model -> aggregate.

In [ ]:
# The sentence we want to tag.
sentence = "Apple opened an office in Paris"

# One call runs tokenize -> model -> aggregate.
entities = ner(sentence)
entities

### b2 — Align spans back onto words

Offsets cash in here. Each item is a dict with the merged `word`, its `entity_group`, a confidence `score`, and char `start`/`end` offsets. We slice the original sentence by those offsets to prove the span lines up with the real text.

In [ ]:
# ---- align spans back onto whole words ----

# Each item is a dict with the merged word, its entity
# group, a confidence score, and char start/end offsets.
for ent in entities:
    word = ent["word"]
    label = ent["entity_group"]
    score = round(float(ent["score"]), 2)
    start, end = ent["start"], ent["end"]

    # Slice the ORIGINAL text to prove the span lines up.
    span = sentence[start:end]

    print(f"{word:<8} {label:<4} {score:.2f}  span={span!r}")

### Expected output

```text
$ python 12_ner.py
Some weights of the model checkpoint at dslim/bert-base-NER were not used...
Apple    ORG  0.99  span='Apple'
Paris    LOC  0.99  span='Paris'

>>> entities
[{'entity_group': 'ORG', 'score': 0.99, 'word': 'Apple',
  'start': 0, 'end': 5},
 {'entity_group': 'LOC', 'score': 0.99, 'word': 'Paris',
  'start': 26, 'end': 31}]
```

## HF13 · Extractive QA: Find the Answer Span

In [ ]:
!pip install -q transformers

### Extractive QA: Find the Answer Span

This notebook builds a question-answering pipeline that doesn't *write* an answer — it points at the exact words inside a context paragraph that answer your question. We extract the span, its confidence score, and the character offsets, then watch the score collapse when the answer isn't there.

### b1 · Build the QA pipeline + inputs

*One pipeline, two strings.* Build a question-answering pipeline pinned to a small SQuAD-tuned model, then define our question and context as plain strings.

In [ ]:
# 13_qa.py
# Extractive QA: point at the answer span inside a context.
# transformers >= 4.44
from transformers import pipeline

# Small, fast QA model fine-tuned on SQuAD.
qa = pipeline(
    task="question-answering",
    model="distilbert-base-cased-distilled-squad",
)

# Input is two plain strings: a question and a context.
question = "When was Linux released?"
context = (
    "Linux is an open-source operating system kernel. "
    "It was first released in 1991 by Linus Torvalds, "
    "a Finnish software engineer."
)

### b2 · Run it & prove the offsets

*Slice the context, it matches.* Call the pipeline, unpack the dict, then slice `context[start:end]` ourselves to prove the character offsets are real.

In [ ]:
# Run the pipeline: it returns a single dict.
result = qa(question=question, context=context)

answer = result["answer"]
score = result["score"]
start = result["start"]
end = result["end"]

print(f"answer: {answer!r}")
print(f"score : {score:.2f}")
print(f"span  : chars {start}-{end}")

# Offsets are real: slice the context ourselves and compare.
sliced = context[start:end]
print(f"slice : {sliced!r}")
assert sliced == answer

### b3 · Reuse it + watch the score drop

*Score tells you when to trust.* Ask a second answerable question, then one the paragraph never answers — the confidence score craters.

In [ ]:
# Same paragraph, a different question — no reload needed.
q2 = "Who created Linux?"
r2 = qa(question=q2, context=context)
print(f"\nq2 answer: {r2['answer']!r}  score: {r2['score']:.2f}")

# A question the context does NOT answer: watch the score drop.
q3 = "What is the price of Linux?"
r3 = qa(question=q3, context=context)
print(f"q3 answer: {r3['answer']!r}  score: {r3['score']:.2f}")

### Expected output

```text
$ python 13_qa.py
answer: 'in 1991'
score : 0.97
span  : chars 142-149
slice : 'in 1991'

q2 answer: 'Linus Torvalds'  score: 0.95
q3 answer: ''  score: 0.04
```

## HF14 · Zero-Shot & Multilingual Pipelines

In [ ]:
!pip install -q transformers torch

This notebook runs **zero-shot text classification**: scoring text against candidate labels you invent yourself, with no training or fine-tuning. We do it first in English, then swap one model string to score French via a multilingual checkpoint.

Candidate labels are *ours* to choose at runtime, not baked into the model weights.

In [ ]:
from transformers import pipeline

# Candidate labels are OURS to choose — not baked into the model.
CANDIDATE_LABELS = ["billing", "support", "sales"]

**b1** — Build the English classifier: `"zero-shot-classification"` wraps an NLI model as a classifier in one line.

In [ ]:
# --- Page 1: English zero-shot --------------------------------
# "zero-shot-classification" wraps an NLI model as a classifier.
classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
)

**b2** — Classify and print ranked scores. The pipeline returns labels sorted best-first, scores aligned — no argmax needed.

In [ ]:
sequence = "I need a refund"
result = classifier(
    sequence,
    candidate_labels=CANDIDATE_LABELS,
    multi_label=False,  # scores normalise to sum to 1.0
)

# pipeline returns labels sorted best-first, scores aligned.
print(f"sequence={result['sequence']!r}")
ranked = zip(result["labels"], result["scores"])
print(" | ".join(f"{lbl} {sc:.2f}" for lbl, sc in ranked))

**b3** — Swap to a multilingual checkpoint. Same call signature, only the model string changes — and now French scores.

In [ ]:
# --- Page 2: multilingual zero-shot ---------------------------
# Same call signature — only the model string changes.
ml_classifier = pipeline(
    "zero-shot-classification",
    model="joeddav/xlm-roberta-large-xnli",
)

fr_sequence = "Je veux un remboursement"
fr_result = ml_classifier(
    fr_sequence,
    candidate_labels=CANDIDATE_LABELS,
    multi_label=False,
)

print(f"sequence={fr_result['sequence']!r}")
fr_ranked = zip(fr_result["labels"], fr_result["scores"])
print(" | ".join(f"{lbl} {sc:.2f}" for lbl, sc in fr_ranked))

### Expected output

```text
$ python 14_zeroshot.py
Device set to use cpu
sequence='I need a refund'
billing 0.91 | support 0.06 | sales 0.03
sequence='Je veux un remboursement'
billing 0.88 | facturation 0.09 | support 0.03
```

## HF15 · Diffusers: Text -> Image

In [ ]:
!pip install -q diffusers transformers accelerate torch pillow

### Diffusers: Text → Image

This notebook hands Stable Diffusion a single prompt — *"a fox in a spacesuit"* — and runs the denoising loop once per seed. Same prompt, four seeds, one guidance scale → four different foxes, tiled into a 2×2 grid. The seed and the guidance scale are the whole game.

### Setup & load
Pick the best device (CUDA if a GPU is present, else CPU) with a matching precision, then pull the pipeline — denoiser plus text encoder — down from the Hub with `from_pretrained` and move it onto the device.

In [ ]:
import torch
from diffusers import StableDiffusionPipeline
from PIL import Image

# Pick the best device + matching precision.
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

# Pull the pipeline (denoiser + text encoder) from the Hub.
MODEL_ID = "runwayml/stable-diffusion-v1-5"
pipe = StableDiffusionPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
)
pipe = pipe.to(device)

### The experiment
Pin the experiment: one prompt, a fixed guidance scale and step count, and a list of four seeds. Every knob is a plain variable at the top, easy to read and easy to change.

In [ ]:
# The experiment: one prompt, four seeds, fixed knobs.
PROMPT = "a fox in a spacesuit"
GUIDANCE = 7.5
STEPS = 25
SEEDS = [42, 43, 44, 45]

### Generation loop
For each seed, build a seeded `torch.Generator` (this is what makes the run reproducible), call the pipeline with the prompt and knobs, pull the single image out, and print a line as each run ticks by.

In [ ]:
# Run the denoising loop once per seed.
images = []
for seed in SEEDS:
    gen = torch.Generator(device=device).manual_seed(seed)
    result = pipe(
        PROMPT,
        generator=gen,
        guidance_scale=GUIDANCE,
        num_inference_steps=STEPS,
    )
    img = result.images[0]
    images.append(img)
    print(f"seed={seed:<3} -> image {len(images)}/{len(SEEDS)} ok")

### Grid & summary
Make a blank canvas two tiles wide and two tiles tall, paste each fox into its quadrant, save the whole thing as one PNG, and print the recap line so the terminal tells the same story as the picture.

In [ ]:
# Tile the four images into a 2x2 grid and save it.
W, H = images[0].size
grid = Image.new("RGB", (W * 2, H * 2))
for i, img in enumerate(images):
    x = (i % 2) * W
    y = (i // 2) * H
    grid.paste(img, (x, y))
grid.save("fox_grid.png")

print(
    f"prompt='{PROMPT}' | {len(images)} images, "
    f"seeds {SEEDS}, guidance={GUIDANCE}"
)

### Expected output

```
$ python 15_diffusers.py
Loading pipeline components...: 100%|██████████| 7/7 [00:04<00:00,  1.59it/s]
seed=42  -> image 1/4 ok
seed=43  -> image 2/4 ok
seed=44  -> image 3/4 ok
seed=45  -> image 4/4 ok
prompt='a fox in a spacesuit' | 4 images, seeds [42, 43, 44, 45], guidance=7.5
```

## HF16 · Whisper ASR: Waveform -> Transcript

In [ ]:
!pip install -q transformers datasets torch

This notebook runs OpenAI's Whisper speech model on a short audio clip and aligns the transcript back to the waveform. We feed it a five-word sample, get the sentence out with one `pipeline` call, then run it again with word-level timestamps so every word is pinned to the slice of sound that produced it.

### Imports & device

GPU if we have one — the encoder is the heavy part. We pin the small English-only checkpoint `whisper-base.en` (74M params, fast, accurate on clean speech).

In [ ]:
# 16_whisper.py
# Run Whisper on an audio clip and align words to the waveform.
# transformers 4.44 · datasets 2.21 · torch 2.4

import torch
from transformers import pipeline
from datasets import load_dataset

# Use the GPU if we have one; the encoder is the heavy part.
device = 0 if torch.cuda.is_available() else -1

# Small English-only checkpoint: 74M params, fast, accurate.
MODEL_ID = "openai/whisper-base.en"

### Build the pipeline

Three tools, one call: this single constructor assembles the feature extractor (makes the spectrogram), the model (encode + decode), and the tokenizer (tokens back to letters).

In [ ]:
# One call assembles feature extractor + model + tokenizer.
asr = pipeline(
    task="automatic-speech-recognition",
    model=MODEL_ID,
    device=device,
)

### Load the audio

The raw float river: load one ready-made speech sample from the Hub and grab its raw float array plus sampling rate — 16 kHz, a couple seconds long.

In [ ]:
# Load one ready-made speech sample from the Hub.
ds = load_dataset(
    "hf-internal-testing/librispeech_asr_dummy",
    "clean",
    split="validation",
)
sample = ds[0]["audio"]
waveform = sample["array"]
sr = sample["sampling_rate"]
print(f"sampling_rate={sr} hz, "
      f"duration={len(waveform) / sr:.1f}s")

### Transcribe

Waveform in, sentence out — no preprocessing we had to write. The result is a dict with one key, `text`, holding the transcript (note Whisper's leading space).

In [ ]:
# Waveform in -> transcript out. No preprocessing by hand.
result = asr(waveform)
print(result)

### Align with timestamps

Every word gets a clock: run again with `return_timestamps="word"` and the result carries a `chunks` list — one entry per word, each with its `text` and a start/end `timestamp` pair. Those spans are the tick marks under the waveform.

In [ ]:
# Run again with word-level timestamps for alignment.
timed = asr(waveform, return_timestamps="word")
for chunk in timed["chunks"]:
    word = chunk["text"]
    start, end = chunk["timestamp"]
    print(f"{word:>9} | {start:.2f}-{end:.2f}s")

### Expected output

```text
sampling_rate=16000 hz, duration=2.3s
{'text': ' hugging face makes models easy'}
  hugging | 0.18-0.62s
     face | 0.62-0.94s
    makes | 0.94-1.31s
   models | 1.31-1.78s
     easy | 1.78-2.20s
```

## HF17 · Accelerate: Throughput & Mixed Precision

In [ ]:
!pip install -q transformers datasets accelerate

This notebook takes a plain PyTorch training loop and wraps it with Hugging Face **Accelerate** so the same code runs in fp16 and across multiple GPUs with no rewrite. We set up a DistilBERT classifier on a slice of IMDB, run one epoch through the Accelerate-wrapped loop, and benchmark the throughput (samples/sec) plus the speedup over a single-GPU fp32 baseline.

**Setup + the fp16 lever.** Import everything, load DistilBERT and its tokenizer, tokenize a 2,000-row slice of IMDB, and build a `DataLoader`. The one new line is `Accelerator(mixed_precision="fp16")` — that single argument turns on mixed precision and the loss scaling behind it.

In [ ]:
import time
import torch
from torch.utils.data import DataLoader
from accelerate import Accelerator
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
)

MODEL = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL, num_labels=2
)

raw = load_dataset("imdb", split="train[:2000]")

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=256)

cols = ["text"]
ds = raw.map(tokenize, batched=True, remove_columns=cols)
ds = ds.rename_column("label", "labels")
ds.set_format("torch")

collator = DataCollatorWithPadding(tokenizer=tokenizer)
loader = DataLoader(ds, batch_size=16, shuffle=True,
                    collate_fn=collator)

# fp16 lever: one argument turns on mixed precision + scaling.
accelerator = Accelerator(mixed_precision="fp16")
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

**The wrapped training loop.** Call `accelerator.prepare` once on the model, optimizer, and dataloader — that moves things to the right devices and wires up the multi-GPU plumbing. The only line changed from vanilla PyTorch is `accelerator.backward(loss)` instead of `loss.backward()`, which does loss scaling and the gradient all-reduce for you. The function returns samples/sec.

In [ ]:
def train_one_epoch(model, optimizer, loader, accelerator):
    """The whole loop. One changed line vs vanilla PyTorch."""
    model, optimizer, loader = accelerator.prepare(
        model, optimizer, loader
    )
    model.train()
    seen = 0
    start = time.time()
    for batch in loader:
        optimizer.zero_grad()
        outputs = model(**batch)
        loss = outputs.loss
        accelerator.backward(loss)   # was loss.backward()
        optimizer.step()
        seen += batch["labels"].shape[0]
    elapsed = time.time() - start
    return seen / elapsed

**Benchmark the throughput.** Run the loop, read the process count and precision off the accelerator, then print one row: samples/sec and the speedup over the 120 smp/s single-GPU fp32 baseline. Launched with a different device count, the same script fills in each row of the table.

In [ ]:
def benchmark():
    """Run the loop and report samples/sec + speedup."""
    n_gpu = accelerator.num_processes
    precision = accelerator.mixed_precision
    throughput = train_one_epoch(
        model, optimizer, loader, accelerator
    )
    base = 120.0  # 1-GPU fp32 reference, samples/sec
    speedup = throughput / base
    tag = f"{n_gpu}x{precision}"
    accelerator.print(
        f"{tag:>10}: {throughput:6.0f} smp/s "
        f"| speedup x{speedup:.2f}"
    )
    return throughput

if __name__ == "__main__":
    benchmark()

### Expected output

```text
$ accelerate launch 17_accelerate.py
Detected 1 process · mixed_precision=fp16
  1xfp16:    210 smp/s | speedup x1.75

### repeated across configs (collected):
  1xfp32:    120 smp/s | speedup x1.00
  1xfp16:    210 smp/s | speedup x1.75
  2xfp16:    390 smp/s | speedup x3.25
  4xfp16:    720 smp/s | speedup x6.00
```

## HF18 · Push to the Hub + a Gradio Space

In [ ]:
!pip install -q transformers huggingface_hub gradio

### Push to the Hub + a Gradio Space

This notebook takes a fine-tuned DistilBERT sentiment classifier from a local folder, pushes it to the Hugging Face Hub with a model card carrying its metrics, then wraps it in a Gradio demo (a Space) anyone can use in a browser.

> Run order: authenticate → push weights/tokenizer → upload the model card → launch the demo. The push steps need a local fine-tuned checkpoint at `./distilbert-sentiment` and write access to your Hub account.

### Imports and config

Gradio for the UI, `login` + `ModelCard` from the Hub, and the transformers loaders. `REPO_ID` is your target Hub repo; `LOCAL_DIR` is the folder the fine-tuning episode left on disk.

In [ ]:
import gradio as gr
from huggingface_hub import login, ModelCard
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    pipeline,
)

REPO_ID = "you/sentiment-distilbert"
LOCAL_DIR = "./distilbert-sentiment"

### b1 — Login + load the checkpoint

Authenticate once (the token is cached locally), then pick up the model and tokenizer we fine-tuned earlier from their local folder.

In [ ]:
# Authenticate once; token is cached locally.
login()

# Load the checkpoint we fine-tuned earlier.
model = AutoModelForSequenceClassification.from_pretrained(LOCAL_DIR)
tokenizer = AutoTokenizer.from_pretrained(LOCAL_DIR)

### b2 — Push model + tokenizer

Three files, one method: `push_to_hub` bundles the weights, config, and tokenizer and uploads them under your username, creating the repo if it doesn't exist.

In [ ]:
# Upload weights, config, and tokenizer to the Hub.
model.push_to_hub(REPO_ID)
tokenizer.push_to_hub(REPO_ID)

### b3 — Write + upload the model card

A YAML header the Hub parses (license + tags → rendered badges) plus markdown with the metrics from the evaluation episode. `ModelCard(...).push_to_hub` drops the README straight into the repo.

In [ ]:
# Build the model card: a YAML header the Hub parses,
# plus markdown with the metrics from the eval episode.
CARD = """\
---
license: apache-2.0
tags:
  - text-classification
  - sentiment
  - distilbert
---

# sentiment-distilbert

DistilBERT fine-tuned for binary sentiment (POS / NEG).

## Metrics

| metric   | value |
|----------|-------|
| accuracy | 0.978 |
| f1       | 0.976 |
"""

# Push the README straight into the repo.
ModelCard(CARD).push_to_hub(REPO_ID)

### b4 — The Gradio demo

Load the model we just pushed back through a `pipeline`, wrap it in a one-line `classify` function, and hand that to `gr.Interface`. `launch()` serves the text box, Submit button, and label — run inside a Hugging Face Space and it becomes a public web app.

In [ ]:
# Load the model we just pushed for the live demo.
pipe = pipeline("sentiment-analysis", model=REPO_ID)

# One tiny function: text in, label + score out.
def classify(text):
    out = pipe(text)[0]
    return {out["label"]: out["score"]}

# Gradio turns that function into a web UI.
demo = gr.Interface(
    fn=classify,
    inputs=gr.Textbox(label="Your text"),
    outputs=gr.Label(label="Sentiment"),
    title="Sentiment DistilBERT",
)

# Run inside a Space → a public web app.
demo.launch()

### Expected output

```text
$ python 18_push_to_hub.py
    Login successful. Token saved to ~/.cache/huggingface/token

model.safetensors: 100%|███████████████| 268M/268M [00:07<00:00, 36.1MB/s]
config.json:       100%|███████████████| 619/619   [00:00<00:00, 412kB/s]
tokenizer.json:    100%|███████████████| 711k/711k  [00:00<00:00, 18.4MB/s]
README.md:         100%|███████████████| 412/412   [00:00<00:00, 287kB/s]

→ model card + Space pushed:
  https://huggingface.co/you/sentiment-distilbert

Running on local URL:  http://127.0.0.1:7860
```

## HF19 · Serving & Inference Endpoints

In [ ]:
!pip install -q huggingface_hub

This notebook calls a deployed Hugging Face Inference Endpoint over HTTP and reads the JSON response. We send one `POST` with `{"inputs": "great product"}`, check the status, time the round trip, and unpack the top prediction into a label and a score.

### Setup: who · which · what
Token says who we are, the repo says which model, the text says what to ask. Never hard-code a token in a shared file.

In [ ]:
"""Call a deployed Inference Endpoint and read the JSON."""
import os
import time

from huggingface_hub import InferenceClient

# --- setup: who, which model, what to ask ---
HF_TOKEN = os.environ["HF_TOKEN"]          # never hard-code a token
MODEL = "you/sentiment"                     # the repo from last episode
TEXT = "great product"                      # the input to classify

### Build & fire the request
One line creates an `InferenceClient` for the model and POSTs the text. Behind it: `POST /models/you/sentiment` with body `{"inputs": text}` and the auth header.

In [ ]:
def call_endpoint(text: str) -> list[dict]:
    """POST the text to the model and return parsed JSON."""
    client = InferenceClient(model=MODEL, token=HF_TOKEN)
    # builds POST /models/you/sentiment with body {"inputs": text}
    return client.text_classification(text)

### Read the status & latency
Wrap the call in a try block so 401 / 422 / 503 land cleanly instead of crashing. Time the call and print status, latency, and a preview of the result.

In [ ]:
def main() -> None:
    start = time.perf_counter()
    try:
        result = call_endpoint(TEXT)
    except Exception as err:                # 401 / 422 / 503 land here
        print(f"POST FAILED | {err}")
        return

    ms = int((time.perf_counter() - start) * 1000)
    print(f"POST 200 | {ms}ms | {result}")

    top = result[0]                         # top prediction
    label = top["label"]
    score = top["score"]
    print(f"-> {label}  ({score:.2f})")

### Run it
The top prediction becomes the one fact a caller wants: this text is positive, and the model is sure.

In [ ]:
if __name__ == "__main__":
    main()

### Expected output

```
$ python 19_inference_endpoint.py
POST 200 | 48ms | [{'label': 'POSITIVE', 'score': 0.99}]
-> POSITIVE  (0.99)
```

## HF20 · Capstone: Fine-Tune -> Evaluate -> Quantize -> Ship

In [ ]:
!pip install -q transformers datasets evaluate bitsandbytes accelerate huggingface_hub

This is the capstone: the full Hugging Face spine end-to-end on **one** model. We take `distilbert-base-uncased`, fine-tune it on IMDB sentiment, evaluate it honestly on the held-out test set, quantize it to 4-bit, and push it to the Hub. No new ideas, just every move from the course run back-to-back in the order you'd actually use them on a real job.

**b1 · Find + Load** — import the five libraries, name the checkpoint and target repo, load the IMDB dataset, and pull the matching tokenizer + model with a fresh 2-label head (`num_labels=2`).

In [ ]:
# 20_capstone.py
# The full Hugging Face spine, end to end, on one model.
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    BitsAndBytesConfig,
)
import evaluate
from huggingface_hub import login

CKPT = "distilbert-base-uncased"
REPO = "you/capstone-model"

# --- Find + Load -----------------------------------------
ds = load_dataset("imdb")
tokenizer = AutoTokenizer.from_pretrained(CKPT)
model = AutoModelForSequenceClassification.from_pretrained(
    CKPT, num_labels=2
)

**b2 · Tokenize + Run once** — map a `truncation=True` tokenizer over the dataset, attach a `DataCollatorWithPadding` so each batch pads to its own longest row, then push one example through the untrained model to prove the pipe breathes (the prediction is random, and that's the point).

In [ ]:
# --- Tokenize --------------------------------------------
def tokenize(batch):
    return tokenizer(batch["text"], truncation=True)

ds_tok = ds.map(tokenize, batched=True)
collator = DataCollatorWithPadding(tokenizer=tokenizer)

# --- Run once (sanity check before training) -------------
probe = tokenizer("this movie ruled", return_tensors="pt")
logits = model(**probe).logits
pred = int(logits.argmax(-1))
print(f"[run ] untrained pred={pred} (random head)")

**b3 · Fine-tune** — `TrainingArguments` sets 3 epochs, batch 16, lr 2e-5, and eval after every epoch; `Trainer` runs the loop with a `compute_metrics` that loads accuracy and F1 from the `evaluate` library.

In [ ]:
# --- Fine-tune -------------------------------------------
args = TrainingArguments(
    output_dir="capstone-out",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    eval_strategy="epoch",
)

acc = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": acc.compute(
            predictions=preds, references=labels
        )["accuracy"],
        "f1": f1.compute(
            predictions=preds, references=labels
        )["f1"],
    }

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_tok["train"],
    eval_dataset=ds_tok["test"],
    data_collator=collator,
    compute_metrics=compute_metrics,
)
trainer.train()

**b3 · Evaluate** — score the fine-tuned model on the held-out test set and print the honest F1 (the number that goes on the model card).

In [ ]:
# --- Evaluate --------------------------------------------
scores = trainer.evaluate()
print(f"[eval] F1={scores['eval_f1']:.3f}")

**b4 · Quantize** — save the fine-tuned weights, then reload them under a `BitsAndBytesConfig(load_in_4bit=True)` and read back the int4 memory footprint.

In [ ]:
# --- Quantize (reload fine-tuned model in 4-bit) ---------
trainer.save_model("capstone-out")
qcfg = BitsAndBytesConfig(load_in_4bit=True)
q_model = AutoModelForSequenceClassification.from_pretrained(
    "capstone-out", quantization_config=qcfg
)
gb = q_model.get_memory_footprint() / 1e9
print(f"[quant] int4 footprint={gb:.1f}GB")

**b4 · Ship** — log in, push the quantized model and the tokenizer to the Hub, and print the spine-complete banner with the F1, the int4 size, and the live URL. (Replace `hf_********` with your own token.)

In [ ]:
# --- Ship ------------------------------------------------
login(token="hf_********")
q_model.push_to_hub(REPO)
tokenizer.push_to_hub(REPO)

print(
    f"SPINE COMPLETE | F1={scores['eval_f1']:.3f} | "
    f"int4={gb:.1f}GB | pushed -> hf.co/{REPO}"
)

### Expected output

```
$ python 20_capstone.py
[run ] untrained pred=0 (random head)
{'loss': 0.6912, 'epoch': 0.0}
{'eval_loss': 0.3104, 'eval_accuracy': 0.9312, 'eval_f1': 0.9330, 'epoch': 1.0}
{'eval_loss': 0.2218, 'eval_accuracy': 0.9618, 'eval_f1': 0.9625, 'epoch': 2.0}
{'eval_loss': 0.1903, 'eval_accuracy': 0.9690, 'eval_f1': 0.9710, 'epoch': 3.0}
{'train_runtime': just shy of 9 min, 'train_samples_per_second': 46.4}
[eval] F1=0.971
[quant] int4 footprint=3.9GB
Uploading: model.safetensors  100%|██████████| 134M/134M [00:06<00:00]
Uploading: tokenizer.json      100%|██████████| 711k/711k [00:00<00:00]
SPINE COMPLETE | F1=0.971 | int4=3.9GB | pushed -> hf.co/you/capstone-model
```